## split BrightDesk with LangChain

In [1]:
from pathlib import Path
from langchain_text_splitters import MarkdownHeaderTextSplitter

kb_path = Path("..")/"practice-kb"/"brightdesk_kb.md"
kb_text = kb_path.read_text(encoding="utf-8")

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "page"), ("##", "section")]
)

chunks = splitter.split_text(kb_text)
print("Chunks:", len(chunks))
print("Type:", type(chunks[0]))
print()
print("metadata:", chunks[0].metadata)
print("page_content:", chunks[0].page_content[:150])

Chunks: 14
Type: <class 'langchain_core.documents.base.Document'>

metadata: {'page': 'BrightDesk IT Solutions — Knowledge Base', 'section': 'Company Overview'}
page_content: BrightDesk IT Solutions is a managed IT services company founded in 2018 and headquartered in Manchester, UK. The company supports around 120 small an


In [2]:
for chunk in chunks:
    if chunk.metadata["section"] == "Employee: Priya Patel":
        print("metadata:", chunk.metadata)
        print("page_content:")
        print(chunk.page_content)

metadata: {'page': 'BrightDesk IT Solutions — Knowledge Base', 'section': 'Employee: Priya Patel'}
page_content:
- Job Title: Head of Service Desk
- Location: Manchester office
- Joined: March 2019
- Responsibilities: Manages the 12-person service desk team, owns the ticketing process and reports monthly on response times.
- Notes: Promoted from Senior Support Engineer to Head of Service Desk in January 2023.


## keep the headings in the text

In [3]:
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "page"), ("##", "section")],
    strip_headers=False,
)

chunks = splitter.split_text(kb_text)

print("Chunks:", len(chunks))
for chunk in chunks:
    if chunk.metadata["section"] == "Employee: Priya Patel":
        print(chunk.page_content)

Chunks: 14
## Employee: Priya Patel
- Job Title: Head of Service Desk
- Location: Manchester office
- Joined: March 2019
- Responsibilities: Manages the 12-person service desk team, owns the ticketing process and reports monthly on response times.
- Notes: Promoted from Senior Support Engineer to Head of Service Desk in January 2023.


## install the embedding and vector database packages

In [4]:
import shutil
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv(override=True)

DB_PATH = "../vector_db"

shutil.rmtree(DB_PATH, ignore_errors=True)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_PATH,
)

print("Stored chunks:", vectorstore._collection.count())

Stored chunks: 14


In [5]:
questions = [
    "How often is my data backed up?",
    "What does Priya do?",
    "Who is in charge of technology?",
]

for q in questions:
    print(q)
    results = vectorstore.similarity_search_with_score(q, k=3)
    for doc, score in results:
        print(f"  {round(score, 3)}  {doc.metadata['section']}")
    print()

How often is my data backed up?
  1.0  Service: CloudGuard
  1.297  Support Hours and Response Times
  1.306  Frequently Asked Questions

What does Priya do?
  0.97  Employee: Priya Patel
  1.418  Employee: Sophie Wright
  1.452  Employee: Rahul Patel

Who is in charge of technology?
  1.008  Employee: Daniel Hughes
  1.258  Policy: Laptops and Devices
  1.378  Employee: Rahul Patel



In [6]:
from langchain_openai import ChatOpenAI

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4.1-nano")

SYSTEM_PROMPT = """You are an assistant for BrightDesk IT Solutions.
Answer questions using ONLY the context provided.
If the context contains several people or services that match the question,
briefly describe each one.
If the answer is not in the context at all, say you don't know."""


def answer_question(question):
    docs = retriever.invoke(question)

    context = ""
    for doc in docs:
        context = context + doc.page_content + "\n\n"

    messages = [
        ("system", SYSTEM_PROMPT),
        ("user", "Context:\n" + context + "\nQuestion: " + question),
    ]

    response = llm.invoke(messages)
    return response.content


print(answer_question("What does Priya do?"))

Priya Patel, the Head of Service Desk at BrightDesk IT Solutions, manages a 12-person service desk team, owns the ticketing process, and reports monthly on response times.


In [7]:
import numpy as np
from sklearn.manifold import TSNE
import plotly.express as px
import nbformat
print(nbformat.__version__)

data = vectorstore._collection.get(include=["embeddings", "metadatas"])

vectors = np.array(data["embeddings"])
titles = []
categories = []
for meta in data["metadatas"]:
    section = meta["section"]
    titles.append(section)
    if ":" in section:
        categories.append(section.split(":")[0])
    else:
        categories.append("Other")

print("Vectors shape:", vectors.shape)

tsne = TSNE(n_components=2, perplexity=5, random_state=42)
points = tsne.fit_transform(vectors)

fig = px.scatter(
    x=points[:, 0],
    y=points[:, 1],
    color=categories,
    hover_name=titles,
    title="BrightDesk chunks in 2D (t-SNE)",
)
fig.show()

5.11.1
Vectors shape: (14, 1536)
